In [ ]:
import pandas as pd

# Load the dataset
file_path = "/content/Online retail.xlsx"
xls = pd.ExcelFile(file_path)

# Check sheet names
xls.sheet_names

# Load the data from the first sheet
df = pd.read_excel(xls, sheet_name="Sheet1")

# Display basic information about the dataset
df.info(), df.head()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                           Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                           --------------  ----- 
 0   shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil  7500 non-null   object
dtypes: object(1)
memory usage: 58.7+ KB


(None,
   shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
 0                             burgers,meatballs,eggs                                                                                                                                                                             
 1                                            chutney                                                                                                                                                                             
 2                                     turkey,avocado                                                                                                                                                                             
 3  mineral water,milk,energy bar,whole wheat rice...                                

In [ ]:
# Split the single column into a list of items for each transaction
df['Transactions'] = df.iloc[:, 0].apply(lambda x: x.split(','))

# Drop the original column with comma-separated values
df = df[['Transactions']]

# Display the first few rows to confirm the transformation
df.head()


,Transactions
0,"[burgers, meatballs, eggs]"
1,[chutney]
2,"[turkey, avocado]"
3,"[mineral water, milk, energy bar, whole wheat ..."
4,[low fat yogurt]


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Convert transactions into a format suitable for one-hot encoding
te = TransactionEncoder()
te_ary = te.fit(df['Transactions']).transform(df['Transactions'])
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# Display the first few rows of the one-hot encoded data
df_encoded.head()


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Manually create a one-hot encoded dataframe
unique_items = set(item for transaction in df['Transactions'] for item in transaction)

# Convert the set of unique items to a list to use as columns
unique_items_list = list(unique_items) # Convert unique_items (a set) to a list

# Create an empty dataframe with items as columns
df_encoded = pd.DataFrame(0, index=range(len(df)), columns=unique_items_list) # Use the list for column names

# Populate the dataframe with binary values (1 if item is in transaction, else 0)
for i, transaction in enumerate(df['Transactions']):
    df_encoded.loc[i,  list(set(transaction))] = 1

# Display the first few rows
df_encoded.head()

,whole wheat pasta,grated cheese,gums,asparagus,mushroom cream sauce,salad,olive oil,mint green tea,zucchini,tomato sauce,...,brownies,carrots,cake,fromage blanc,eggplant,soda,spaghetti,dessert wine,corn,salmon
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Convert the set of unique items to a sorted list
unique_items = sorted(list(set(item for transaction in df['Transactions'] for item in transaction)))

# Create an empty dataframe with items as columns
df_encoded = pd.DataFrame(0, index=range(len(df)), columns=unique_items)

# Populate the dataframe with binary values (1 if item is in transaction, else 0)
for i, transaction in enumerate(df['Transactions']):
    df_encoded.loc[i, list(set(transaction))] = 1

# Display the first few rows
df_encoded.head()


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Check for duplicate items in transactions
from collections import Counter

# Count occurrences of each item across all transactions
item_counts = Counter(item for transaction in df['Transactions'] for item in transaction)

# Find duplicate item names (if any)
duplicates = [item for item, count in item_counts.items() if count > 1]

# Display the first few duplicate items (if any)
duplicates[:10]


['burgers',
 'meatballs',
 'eggs',
 'chutney',
 'turkey',
 'avocado',
 'mineral water',
 'milk',
 'energy bar',
 'whole wheat rice']

In [ ]:
# Create an empty dataframe with items as columns
df_encoded = pd.DataFrame(0, index=range(len(df)), columns=unique_items, dtype=int)

# Populate the dataframe with binary values (1 if item is in transaction, else 0)
for i, transaction in enumerate(df['Transactions']):
    for item in transaction:
        df_encoded.at[i, item] = 1

# Display the first few rows
df_encoded.head()


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Apply the Apriori algorithm to find frequent itemsets
frequent_itemsets = apriori(df_encoded, min_support=0.02, use_colnames=True)

# Generate association rules with confidence and lift thresholds
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Display the top rules sorted by lift
rules.sort_values(by="lift", ascending=False).head()


/usr/local/lib/python3.11/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
67,(ground beef),(spaghetti),0.098267,0.174133,0.039200,0.398915,2.290857,1.0,0.022088,1.373959,0.624888,0.168096,0.272176,0.312015
66,(spaghetti),(ground beef),0.174133,0.098267,0.039200,0.225115,2.290857,1.0,0.022088,1.163699,0.682292,0.168096,0.140672,0.312015
89,(olive oil),(spaghetti),0.065733,0.174133,0.022933,0.348884,2.003547,1.0,0.011487,1.268387,0.536127,0.105716,0.211597,0.240292
88,(spaghetti),(olive oil),0.174133,0.065733,0.022933,0.131700,2.003547,1.0,0.011487,1.075972,0.606497,0.105716,0.070608,0.240292
80,(soup),(mineral water),0.050533,0.238267,0.023067,0.456464,1.915771,1.0,0.011026,1.401441,0.503458,0.086804,0.286449,0.276637


In [ ]:
from itertools import combinations

# Define minimum support threshold
min_support = 0.02

# Compute support for individual items
item_support = df_encoded.mean()

# Filter items that meet the support threshold
frequent_items = item_support[item_support >= min_support]

# Generate frequent itemsets of size 2 (pairs of items)
frequent_itemsets = []
for itemset in combinations(frequent_items.index, 2):
    support = (df_encoded[list(itemset)].sum(axis=1) == 2).mean()
    if support >= min_support:
        frequent_itemsets.append((itemset, support))

# Convert to DataFrame
frequent_itemsets_df = pd.DataFrame(frequent_itemsets, columns=["Itemset", "Support"])

# Display the top frequent itemsets
frequent_itemsets_df.sort_values(by="Support", ascending=False).head()


,Itemset,Support
44,"(mineral water, spaghetti)",0.059733
12,"(chocolate, mineral water)",0.052667
20,"(eggs, mineral water)",0.050933
38,"(milk, mineral water)",0.048000
35,"(ground beef, mineral water)",0.040933


In [ ]:
# Function to compute confidence
def compute_confidence(itemset, support, item_support):
    item1, item2 = itemset
    conf_1to2 = support / item_support[item1]  # Confidence of item1 → item2
    conf_2to1 = support / item_support[item2]  # Confidence of item2 → item1
    return conf_1to2, conf_2to1

# Function to compute lift
def compute_lift(support, item_support, itemset):
    item1, item2 = itemset
    lift = support / (item_support[item1] * item_support[item2])
    return lift

# Compute confidence and lift for each frequent itemset
rules_list = []
for itemset, support in frequent_itemsets:
    conf_1to2, conf_2to1 = compute_confidence(itemset, support, item_support)
    lift = compute_lift(support, item_support, itemset)
    rules_list.append((itemset[0], itemset[1], support, conf_1to2, conf_2to1, lift))

# Convert to DataFrame
rules_df = pd.DataFrame(rules_list, columns=["Antecedent", "Consequent", "Support", "Confidence (A→B)", "Confidence (B→A)", "Lift"])

# Filter rules based on meaningful thresholds
min_confidence = 0.3
min_lift = 1.0
filtered_rules = rules_df[(rules_df["Confidence (A→B)"] >= min_confidence) & (rules_df["Lift"] >= min_lift)]

# Display the top rules sorted by lift
filtered_rules.sort_values(by="Lift", ascending=False).head()


,Antecedent,Consequent,Support,Confidence (A→B),Confidence (B→A),Lift
36,ground beef,spaghetti,0.039200,0.398915,0.225115,2.290857
47,olive oil,spaghetti,0.022933,0.348884,0.131700,2.003547
0,burgers,eggs,0.028800,0.330275,0.160237,1.837585
35,ground beef,mineral water,0.040933,0.416554,0.171796,1.748266
14,cooking oil,mineral water,0.020133,0.394256,0.084499,1.654683


**Insights:**

* Customers buying ground beef often buy spaghetti, indicating a strong correlation.
* Olive oil and spaghetti are commonly purchased together, possibly for pasta dishes.
* Burgers and eggs suggest meal combinations.
* Mineral water is a frequent add-on for multiple products.

# **INTERVIEW QUESTION**
**1. What is Lift and Why is it Important in Association Rules?**

 Answer: Lift measures how much more likely two items are purchased together compared to if they were independent.

  Formula:
  Lift=Confidence(A→B)/Support(B)
  
  Importance:

 * Lift > 1 → Strong positive association (items frequently bought together).
 * Lift = 1 → No relationship (independent purchases).
 * Lift < 1 → Negative association (items rarely bought together).

  Example: If Lift(Milk → Bread) = 2.5, it means customers who buy Milk are 2.5 times more likely to buy Bread than random chance.

**2. What is Support and Confidence? How Do You Calculate Them?**

Answer:
 Support (How often an item appears in transactions)

  Formula:
   Support(A)=Transactions containing A/Total transactions
    
  Example: If Milk appears in 300 out of 1000 transactions:
      
  Support(Milk) = 300 / 1000= 0.3 (30%)

  Confidence (How often B is bought when A is bought)

  Confidence(A → B) = Support(A ∩ B)/ Support(A)

  Example: If 200 transactions have {Milk, Bread} and 300 have Milk:

  Confidence(Milk → Bread)=200/300=0.67(67%)

Interpretation: 67% of the time, when Milk is bought, Bread is also bought.

**3. What are Some Limitations or Challenges of Association Rule Mining?**
💡 Answer:

 1. Large Number of Rules:

 Can generate too many rules, making it hard to extract meaningful insights.

 Solution: Use Lift or filter by high support & confidence.
 2. High Computational Cost:

 Apriori algorithm scans the dataset multiple times, making it slow.

 Solution: Use FP-Growth to improve efficiency.
 3. Choosing the Right Support & Confidence Thresholds:

 Too high → Miss important patterns.

 Too low → Get many irrelevant rules.

 Solution: Trial and error with domain knowledge.
 4. No Temporal Information:

 Does not consider purchase sequences (e.g., buying a phone, then a case).

 Solution: Use Sequential Pattern Mining instead.
 5. Rare but Valuable Associations are Missed:

 High-value items (e.g., cars) may have low support but are still important.

 Solution: Consider Lift to capture such insights.
